**Creation of datacube with dim and fact**

In [0]:
CREATE OR REPLACE TABLE 03_prod_gold.models.sales_analytics AS

SELECT
    f.order_number,
    f.order_date,

    COALESCE(d.year, -1) AS year,
    COALESCE(d.month, -1) AS month,
    COALESCE(d.month_name, 'Unknown') AS month_name,

    COALESCE(f.customer_key, -1) AS customer_key,
    COALESCE(c.gender, 'Unknown') AS gender,
    COALESCE(c.continent, 'Unknown') AS continent,

    COALESCE(f.product_key, -1) AS product_key,
    COALESCE(p.category, 'Unknown') AS category,
    COALESCE(p.subcategory, 'Unknown') AS subcategory,

 
    COALESCE(f.store_key, -1) AS store_key,
    COALESCE(s.country, 'Unknown') AS country,

    COALESCE(f.quantity, 0) AS quantity,
    COALESCE(f.revenue_usd, 0) AS revenue_usd,
    COALESCE(f.delivery_days, 0) AS delivery_days

FROM 03_prod_gold.models.fact_sales f

LEFT JOIN 03_prod_gold.models.dim_customers c
    ON f.customer_key = c.customer_key

LEFT JOIN 03_prod_gold.models.dim_products p
    ON f.product_key = p.product_key

LEFT JOIN 03_prod_gold.models.dim_stores s
    ON f.store_key = s.store_key

LEFT JOIN 03_prod_gold.models.dim_date d
    ON f.order_date = d.order_date;

**1. Monthly Revenue Trend - Merge Sales, Products, Exchange Rates. Calculate total revenue USD by month (2024)**

In [0]:
SELECT 
    year,
    month,
    SUM(revenue_usd) AS revenue_usd
FROM 03_prod_gold.models.sales_analytics
WHERE year = 2020
GROUP BY year, month
ORDER BY month;

**2. Peak Month Analysis - From Q1, identify top 3 revenue months. Calculate % of annual total**


In [0]:
WITH monthly AS (
    SELECT 
        month,
        SUM(revenue_usd) AS revenue
    FROM 03_prod_gold.models.sales_analytics
    WHERE year = 2020
    GROUP BY month
),
total AS (
    SELECT SUM(revenue) AS total_rev FROM monthly
)

SELECT 
    m.month,
    m.revenue,
    (m.revenue / t.total_rev) * 100 AS pct_of_total
FROM monthly m
CROSS JOIN total t
ORDER BY m.revenue DESC
LIMIT 3;

**3. Holiday Drivers - For Q2 peak months, show top 3 product categories driving revenue**


In [0]:
WITH peak_months AS (
    SELECT month
    FROM (
        SELECT month, SUM(revenue_usd) AS revenue
        FROM 03_prod_gold.models.sales_analytics
        WHERE year = 2020
        GROUP BY month
        ORDER BY revenue DESC
        LIMIT 3
    )
),
category_sales AS (
    SELECT 
        category,
        SUM(revenue_usd) AS revenue
    FROM 03_prod_gold.models.sales_analytics
    WHERE month IN (SELECT month FROM peak_months)
    GROUP BY category
),
total AS (
    SELECT SUM(revenue) AS total_rev FROM category_sales
)

SELECT 
    c.category,
    c.revenue,
    (c.revenue / t.total_rev) * 100 AS pct_of_peak
FROM category_sales c
CROSS JOIN total t
ORDER BY revenue DESC
LIMIT 3;

**4. Delivery Performance - Calculate overall avg delivery time across all orders**

In [0]:
SELECT 
    AVG(delivery_days) AS average_days,
    COUNT(DISTINCT order_number) AS total_orders
FROM 03_prod_gold.models.sales_analytics;

**5. Country Delivery Issue - Avg delivery time by store country. Show slowest 5 countries**

In [0]:
SELECT 
    country,
    AVG(delivery_days) AS avg_days,
    COUNT(order_number) AS order_count,
    PERCENTILE_APPROX(delivery_days, 0.5) AS median_days
FROM 03_prod_gold.models.sales_analytics
WHERE store_key=0
GROUP BY country
ORDER BY avg_days DESC
LIMIT 5;

**6. Channel Performance - AOV (revenue/orders) by Online vs In-Store across continents (Online = StoreKey.isna())**

In [0]:
SELECT 
    continent,

    SUM(CASE WHEN store_key = 0 THEN revenue_usd END) /
    COUNT(DISTINCT CASE WHEN store_key = 0 THEN order_number END) AS AOV_online,

    SUM(CASE WHEN store_key != 0 THEN revenue_usd END) /
    COUNT(DISTINCT CASE WHEN store_key != 0 THEN order_number END) AS AOV_store,

    COUNT(DISTINCT CASE WHEN store_key = 0 THEN order_number END) AS online_orders,
    COUNT(DISTINCT CASE WHEN store_key != 0 THEN order_number END) AS store_orders

FROM 03_prod_gold.models.sales_analytics
GROUP BY continent HAVING continent!='Unknown';

**7. Volume Leaders - Top 5 product categories by total units sold**

In [0]:
WITH total AS (
    SELECT SUM(quantity) AS total_units
    FROM 03_prod_gold.models.sales_analytics
)

SELECT 
    RANK() OVER (ORDER BY SUM(quantity) DESC) AS rank,
    category,
    SUM(quantity) AS units_sold,
    (SUM(quantity) / t.total_units) * 100 AS pct_of_total
FROM 03_prod_gold.models.sales_analytics
CROSS JOIN total t
GROUP BY category, t.total_units
ORDER BY units_sold DESC
LIMIT 5;

**8. Revenue Leaders - Top 5 product categories by total revenue USD**

In [0]:
WITH total AS (
    SELECT SUM(revenue_usd) AS total_revenue
    FROM 03_prod_gold.models.sales_analytics
)

SELECT 
    RANK() OVER (ORDER BY SUM(revenue_usd) DESC) AS rank,
    category,
    SUM(revenue_usd) AS revenue_usd,
    (SUM(revenue_usd) / t.total_revenue) * 100 AS pct_of_total
FROM 03_prod_gold.models.sales_analytics
CROSS JOIN total t
GROUP BY category, t.total_revenue
ORDER BY revenue_usd DESC
LIMIT 5;

**9. Customer Profile - Customer count and spending by Continent × Gender**

In [0]:
SELECT 
    continent,
    gender,
    COUNT(DISTINCT customer_key) AS customer_count,
    SUM(revenue_usd) AS total_spend,
    SUM(revenue_usd) / COUNT(DISTINCT customer_key) AS avg_spend_per_customer
FROM 03_prod_gold.models.sales_analytics
GROUP BY continent, gender HAVING continent!="Unknown";

**10. Customer Loyalty - Repeat customer rate (% with 2+ orders) by continent**

In [0]:
WITH customer_orders AS (
    SELECT 
        continent,
        customer_key,
        COUNT(DISTINCT order_number) AS orders
    FROM 03_prod_gold.models.sales_analytics
    GROUP BY continent, customer_key
)

SELECT 
    continent,
    COUNT(DISTINCT CASE WHEN orders >= 2 THEN customer_key END) * 100.0 /
    COUNT(DISTINCT customer_key) AS repeat_rate_pct,
    COUNT(DISTINCT customer_key) AS unique_customers,
    COUNT(DISTINCT CASE WHEN orders >= 2 THEN customer_key END) AS repeat_customers
FROM customer_orders
GROUP BY continent HAVING continent!='Unknown';